In [ ]:
# Create Geolife file
from datetime import date
import os
import pandas as pd
from src.calculate_speed import calculate_speed_for_dataframe, haversine
from src.optimized_analysis import get_prediction_for_trip
from src.process_files_geolife import labels_iterrable, get_filename_for_label_row
from src.stats import is_correct_prediction_custom_thresholds

today = date.today().strftime("%Y%m%d")

root_path = 'data/Geolife/'
output_file_path = f'{root_path}{today}_geolife_database_metrics.csv'

rows = []
count = 0
total_observations_count = 0
for directory in sorted([f for f in os.listdir(root_path) if f.isdigit() and len(f) == 3]):
    user_path = f"{root_path}{directory}/"
    trajectory_path = f'{user_path}Processed_Trajectory/'
    labels_path = f'{user_path}labels.txt'

    for index, row in labels_iterrable(labels_path):
        trajectory_file = get_filename_for_label_row(row, trajectory_path)
        file_to_process = f"{trajectory_path}{trajectory_file}"
        trip_id = f"{directory}.{int(pd.to_datetime(row['Start Time']).strftime('%Y%m%d%H%M%S'))}"

        df = pd.read_csv(file_to_process,
                     skiprows=6,
                     names=['lat', 'lng', '0', 'alt', 'days_since_1899', 'date', 'time'])
        if 'timestamp' not in df.columns:
            df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'])
            df = df.sort_values('timestamp')
            # df = df.set_index('timestamp')

        # Sparsity metric
        datapoints_bucketed_by_minute = df.set_index('timestamp').resample('5s').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        datapoints_bucketed_by_minute = df.set_index('timestamp').resample('20s').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_20s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        datapoints_bucketed_by_minute = df.set_index('timestamp').resample('30s').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        datapoints_bucketed_by_minute = df.set_index('timestamp').resample('1min').size()
        num_empty = (datapoints_bucketed_by_minute == 0).sum()
        total_buckets = len(datapoints_bucketed_by_minute)
        sparsity_60s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

        total_distance_km = haversine(
            df['lat'].shift(1), df['lng'].shift(1),
            df['lat'], df['lng']
        ).sum()
        _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)


        number_of_records = len(df)
        if df.empty:
            total_trip_time_minutes = 0.0
            density_records_per_minute = 0.0
        else:
            total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
            density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0

        prediction = get_prediction_for_trip(df)
        actual_mode = row['Transportation Mode']
        is_prediction_correct = is_correct_prediction_custom_thresholds(prediction, actual_mode)

        count += 1
        total_observations_count += len(df)
        print(f"Appending row for {trip_id} ({count} files processed)")
        rows.append({
            'trip_id': trip_id,
            'number_of_records': number_of_records,
            'total_trip_time_minutes': round(total_trip_time_minutes, 3),
            'total_distance_km': round(total_distance_km, 3),
            'average_speed_kmh': round(average_speed_kmh, 3),
            'max_speed_kmh': round(max_speed_kmh, 3),
            'density_records_per_minute': round(density_records_per_minute, 3),
            'sparsity_5s': round(sparsity_5s, 3),
            'sparsity_20s': round(sparsity_20s, 3),
            'sparsity_30s': round(sparsity_30s, 3),
            'sparsity_60s': round(sparsity_60s, 3),
            'prediction': prediction,
            'actual_mode': actual_mode,
            'is_prediction_correct': is_prediction_correct
        })

dataframe = pd.DataFrame(rows)
dataframe = dataframe.set_index('trip_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f'Total Observations: {total_observations_count}')

# 064.20080831161510
# 065.20110824135121

In [ ]:
# Create rMove database metrics file
from datetime import date
import os
import pandas as pd
from src.calculate_speed import calculate_speed_for_dataframe, haversine
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.stats import is_correct_prediction_custom_thresholds

today = date.today().strftime("%Y%m%d")
root_path = 'data/rMove/'
output_file_path = f'{root_path}{today}_rmove_database_metrics.csv'

locations_df = pd.read_csv(f'{root_path}Location_2023.csv')
trips_df = pd.read_csv(f'{root_path}Household_Travel_Survey_Trips_-7221806773183684102.csv', low_memory=False)
trips_df = trips_df.set_index('trip_id')

rows = []
count = 0

for trip_id, df in locations_df.groupby('tripid'):
    if trip_id not in trips_df.index:
        continue

    df = df.rename(columns={'lon': 'lng', 'collect_time': 'timestamp'})
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp')

    # Sparsity metric
    datapoints_bucketed_by_minute = df.set_index('timestamp').resample('5s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    datapoints_bucketed_by_minute = df.set_index('timestamp').resample('20s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_20s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    datapoints_bucketed_by_minute = df.set_index('timestamp').resample('30s').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    datapoints_bucketed_by_minute = df.set_index('timestamp').resample('1min').size()
    num_empty = (datapoints_bucketed_by_minute == 0).sum()
    total_buckets = len(datapoints_bucketed_by_minute)
    sparsity_60s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

    total_distance_km = haversine(
        df['lat'].shift(1), df['lng'].shift(1),
        df['lat'], df['lng']
    ).sum()
    _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

    # Calculate density
    number_of_records = len(df)
    if not df.empty:
        total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
        density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
    else:
        total_trip_time_minutes = 0.0
        density_records_per_minute = 0.0

    # Generate prediction and compare to actual mode
    prediction = get_prediction_for_trip_rmove(df)
    trip_info = trips_df.loc[trip_id]
    actual_mode = trip_info['mode_1']
    mapped_prediction = map_to_shared_mode_names(prediction)
    mapped_actual = map_to_shared_mode_names(actual_mode)
    is_prediction_correct = is_correct_prediction_custom_thresholds(mapped_prediction, mapped_actual)

    count += 1
    print(f"Appending row for {trip_id} ({count} files processed)")
    rows.append({
        'trip_id': trip_id,
        'number_of_records': number_of_records,
        'total_trip_time_minutes': round(total_trip_time_minutes, 3),
        'total_distance_km': round(total_distance_km, 3),
        'density_records_per_minute': round(density_records_per_minute, 3),
        'average_speed_kmh': round(average_speed_kmh, 3),
        'max_speed_kmh': round(max_speed_kmh, 3),
        'sparsity_5s': round(sparsity_5s, 3),
        'sparsity_20s': round(sparsity_20s, 3),
        'sparsity_30s': round(sparsity_30s, 3),
        'sparsity_60s': round(sparsity_60s, 3),
        'prediction': mapped_prediction,
        'actual_mode': mapped_actual,
        'is_prediction_correct': is_prediction_correct
    })

dataframe = pd.DataFrame(rows)
dataframe = dataframe.set_index('trip_id')
dataframe.to_csv(output_file_path, index=True, header=True, mode='w')

print("Done!")
print(f"Trips processed: {count}")

In [7]:
# Generate Spectus database metrics for 2000 users
from datetime import date
import logging
import pandas as pd
from pathlib import Path
from src.filters import should_filter_out_trip_due_to_abnormality
from src.map_terminology import map_to_shared_mode_names
from src.optimized_analysis import get_prediction_for_trip_rmove
from src.calculate_speed import calculate_speed_for_dataframe, haversine

today = date.today().strftime("%Y%m%d")

root_path = 'data/Spectus/Lyra_Processed/'
input_path = f'{root_path}split_by_user/'
output_file_path = f'{root_path}{today}_spectus_database_metrics.csv'
filtered_trips_path = f'{root_path}{today}_filtered_trips.csv'

logging.basicConfig(
    filename=f'{root_path}{today}_error_log.log',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

user_count = 0
total_trips_count = 0
filtered_trips_count = 0
for file in Path(input_path).iterdir():
    if not file.is_file():
        continue

    count = 0
    try:
        print(f'Processing user {user_count}')
        locations_df = pd.read_csv(f'{input_path}{file.name}', low_memory=False)

        # Remove stop points
        locations_df = locations_df[locations_df['traj_id'] != -99]

        locations_df['traj_id'] = locations_df['user_ID'].astype(str) + '_' + locations_df['traj_id'].astype(str)

        rows = []
        for traj_id, df in locations_df.groupby('traj_id'):

            df = df.rename(columns={'orig_lat': 'lat', 'orig_long': 'lng', 'datetime': 'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'])
            df = df.sort_values('timestamp')

            # Sparsity metric
            datapoints_bucketed_by_minute = df.set_index('timestamp').resample('5s').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_5s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            datapoints_bucketed_by_minute = df.set_index('timestamp').resample('20s').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_20s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            datapoints_bucketed_by_minute = df.set_index('timestamp').resample('30s').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_30s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            datapoints_bucketed_by_minute = df.set_index('timestamp').resample('1min').size()
            num_empty = (datapoints_bucketed_by_minute == 0).sum()
            total_buckets = len(datapoints_bucketed_by_minute)
            sparsity_60s = (num_empty / total_buckets) if total_buckets > 0 else 1.0

            # Calculate density
            number_of_records = len(df)
            if not df.empty:
                total_trip_time_minutes = (df['timestamp'].iloc[-1] - df['timestamp'].iloc[0]).total_seconds() / 60
                density_records_per_minute = number_of_records / total_trip_time_minutes if total_trip_time_minutes > 0 else 0.0
            else:
                total_trip_time_minutes = 0.0
                density_records_per_minute = 0.0

            total_distance_km = haversine(
                df['lat'].shift(1), df['lng'].shift(1),
                df['lat'], df['lng']
            ).sum()
            _, average_speed_kmh, max_speed_kmh = calculate_speed_for_dataframe(df, with_smoothing=False)

            # Generate prediction and compare to actual mode
            prediction = get_prediction_for_trip_rmove(df)
            mapped_prediction = map_to_shared_mode_names(prediction)

            trip_details = {
                'trip_id': traj_id,
                'number_of_records': number_of_records,
                'total_trip_time_minutes': round(total_trip_time_minutes, 3),
                'total_distance_km': round(total_distance_km, 3),
                'average_speed_kmh': round(average_speed_kmh, 3),
                'max_speed_kmh': round(max_speed_kmh, 3),
                'density_records_per_minute': round(density_records_per_minute, 3),
                'sparsity_5s': round(sparsity_5s, 3),
                'sparsity_20s': round(sparsity_20s, 3),
                'sparsity_30s': round(sparsity_30s, 3),
                'sparsity_60s': round(sparsity_60s, 3),
                'prediction': mapped_prediction
            }

            if should_filter_out_trip_due_to_abnormality(trip_details):
                filtered_trip_dataframe = pd.DataFrame([trip_details])
                filtered_trip_dataframe = filtered_trip_dataframe.set_index('trip_id')
                if filtered_trips_count == 0:
                    filtered_trip_dataframe.to_csv(filtered_trips_path, index=True, header=True, mode='w')
                else:
                    filtered_trip_dataframe.to_csv(filtered_trips_path, index=True, header=False, mode='a')

                filtered_trips_count += 1
                continue

            rows.append(trip_details)
            count += 1

        dataframe = pd.DataFrame(rows)
        dataframe = dataframe.set_index('trip_id')
        if user_count == 0:
            dataframe.to_csv(output_file_path, index=True, header=True, mode='w')
        else:
            dataframe.to_csv(output_file_path, index=True, header=False, mode='a')
    except Exception as e:
        print(f"Unexpected error processing file: {file.name}")
        print(f"Error: {e}")
        logging.error(f"Unexpected error processing file: {file.name}")
        logging.error(f"Error: {e}")

    user_count += 1
    print(f'Appended {count} rows')
    total_trips_count += count

print("Done!")
print(f"Total trips processed: {total_trips_count}")
print(f"Trips filtered: {filtered_trips_count}")

Processing user 0
Appended 44 rows
Processing user 1
Appended 357 rows
Processing user 2
Appended 386 rows
Processing user 3
Appended 354 rows
Processing user 4
Appended 303 rows
Processing user 5
Appended 1275 rows
Processing user 6
Appended 178 rows
Processing user 7
Appended 345 rows
Processing user 8
Appended 107 rows
Processing user 9
Appended 298 rows
Processing user 10
Appended 94 rows
Processing user 11
Appended 321 rows
Processing user 12
Appended 142 rows
Processing user 13
Appended 1047 rows
Processing user 14
Appended 52 rows
Processing user 15
Appended 362 rows
Processing user 16
Appended 353 rows
Processing user 17
Appended 619 rows
Processing user 18
Appended 509 rows
Processing user 19
Appended 105 rows
Processing user 20
Appended 75 rows
Processing user 21
Appended 362 rows
Processing user 22
Appended 16 rows
Processing user 23
Unexpected error processing file: user_04520316017702f46a82c5bcb2443cb32b072241b269da0e96eb29334a3b0865.csv
Error: "None of ['trip_id'] are in 

/Users/zunta/.pyenv/versions/mode_analysis/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4598: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,


Appended 316 rows
Processing user 652
Appended 700 rows
Processing user 653
Appended 332 rows
Processing user 654
Appended 31 rows
Processing user 655
Appended 1 rows
Processing user 656
Appended 183 rows
Processing user 657
Appended 248 rows
Processing user 658
Appended 72 rows
Processing user 659
Appended 191 rows
Processing user 660
Appended 88 rows
Processing user 661
Appended 485 rows
Processing user 662
Appended 181 rows
Processing user 663
Appended 308 rows
Processing user 664
Appended 133 rows
Processing user 665
Appended 506 rows
Processing user 666
Appended 79 rows
Processing user 667
Unexpected error processing file: user_6c9c7d71ff9e394973dd9df599835c46113c788d607c1c5496d48e2dc10220a3.csv
Error: "None of ['trip_id'] are in the columns"
Appended 0 rows
Processing user 668
Appended 413 rows
Processing user 669
Appended 73 rows
Processing user 670
Appended 26 rows
Processing user 671
Appended 42 rows
Processing user 672
Appended 50 rows
Processing user 673
Unexpected error pro

/Users/zunta/.pyenv/versions/mode_analysis/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4598: RuntimeWarning: invalid value encountered in subtract
  subtract(b, diff_b_a * (1 - t), out=lerp_interpolation, where=t >= 0.5,


Appended 128 rows
Processing user 1191
Appended 26 rows
Processing user 1192
Appended 243 rows
Processing user 1193
Appended 684 rows
Processing user 1194
Appended 316 rows
Processing user 1195
Appended 526 rows
Processing user 1196
Appended 234 rows
Processing user 1197
Unexpected error processing file: user_1b766a575bd88123e8cefc58052a84324582913a9377a13ab987c7c65bb06ed8.csv
Error: "None of ['trip_id'] are in the columns"
Appended 0 rows
Processing user 1198
Appended 225 rows
Processing user 1199
Appended 397 rows
Processing user 1200
Appended 490 rows
Processing user 1201
Appended 386 rows
Processing user 1202
Appended 257 rows
Processing user 1203
Appended 321 rows
Processing user 1204
Appended 132 rows
Processing user 1205
Appended 41 rows
Processing user 1206
Appended 96 rows
Processing user 1207
Appended 146 rows
Processing user 1208
Appended 654 rows
Processing user 1209
Appended 111 rows
Processing user 1210
Appended 243 rows
Processing user 1211
Appended 84 rows
Processing us